# EP02 — Pandas para plantas de beneficio
## Metalurgia + Data Science**Dataset:** `Dataset_Planta_B_EP02.csv` — circuito selectivo Pb-Zn, 2568 turnos, 2023-2025.
Este notebook se construye durante el video. Las celdas de código están vacías a propósito:la idea es que lo escribas conmigo, no que lo leas.
> Si te pierdes, pausa. El notebook resuelto no sustituye haberlo tecleado.

---
## Preparación
Carga del dataset. Si trabajas en Colab, sube el archivo o apunta a la URL cruda de GitHub.

---
# Bloque 1 — ¿Qué me llegó?
Te pasan un CSV de un circuito Pb-Zn y no lo has visto nunca. ¿Por dónde empiezas?

### 1.1 ¿De qué tamaño es?

### 1.2 ¿Cómo se ven los datos?

### 1.3 ¿Qué tipo de dato tiene cada columna? Ojo con `Fecha`: llega como texto. Y el formato es día/mes/año — si dejas que pandas adivine, te voltea los meses con los días en silencio.

### 1.4 ¿En qué rangos se mueve cada variable?

### 1.5 ¿Qué significa cada columna?
El nombre de cada columna sigue un patrón. Descifrarlo es el primer trabajo metalúrgico del episodio — y no lo hace pandas, lo haces tú.

### 1.6 ¿Me llegó completo todo?

**¿Qué te dice la *forma* de los faltantes?**
No los cuentes nada más. Fíjate en cómo se agrupan. La respuesta la trabajamos en el Bloque 3.

---
# Bloque 2 — ¿Cuál es tu umbral?
Pregunta de planta: ¿en qué turnos mi concentrado de plomo salió sucio?
> **Principio de este bloque:** los umbrales no van escritos dentro del filtro. Van como variables arriba de la celda.

### 2.1 Quedarme solo con las columnas de calidad de concentrado

### 2.2 ¿Cómo se comportan los contaminantes?

### 2.3 Filtrar por umbral — cámbialo por el tuyo

**Prueba otros valores.** Cambia `UMBRAL_ZN` y vuelve a correr. ¿Cuántos turnos marca cada corte?

### 2.4 Dos condiciones a la vez
Cada condición va entre paréntesis. Se usa `&` y `|`, no `and` / `or`.

### 2.5 Lo mismo con `query()`
El `@` permite usar una variable de Python dentro de la consulta.

### 2.6 `loc` vs `iloc` — el error que todos cometen
Filtramos con un corte más selectivo para que se vea bien lo que pasa con el índice.

El índice **conservó las etiquetas originales** y quedó lleno de huecos. Ya no es 0, 1, 2, 3... Aquí es donde se rompe todo.

Ahora pide `loc[5]` pensando en "la sexta fila".

**Ninguna de las dos tiene error.** Las dos devuelven datos válidos. Y son turnos distintos. Un error que se detiene te obliga a revisar. Este no se detiene: te entrega el turno equivocado y tú sigues trabajando.`loc` pregunta por nombre. `iloc` pregunta por posición.

---
# Bloque 3 — ¿Cuánto recuperé?
**Fórmula de dos productos:**$$R = 100 \cdot \frac{c\,(f - t)}{f\,(c - t)}$$donde `f` = ley de alimentación, `c` = ley de concentrado, `t` = ley de cola.

### 3.1 Recuperación del circuito de plomo

### 3.2 Ahora el zinc

Mismo procedimiento. ¿Qué usarías como alimentación del circuito de zinc?

### 3.3 Revisa el diagrama de flujo antes de aceptar ese número

Este es un circuito **secuencial selectivo**. Antes de que el mineral llegue al circuito
de zinc, ya pasó por flotación de plomo — y parte del zinc se fue en ese concentrado.

Para llevar la recuperación de zinc a base de cabeza necesitas saber **cuánta masa
se fue en cada concentrado**. Eso es el *mass pull*.

### 3.4 Recuperación de zinc en base a cabeza

**13.7 puntos de recuperación.**

No son dos fórmulas distintas: son dos **bases de cálculo** distintas. La primeraignora todo el zinc que se fue en el concentrado de plomo — el `Conc_Pb_Zn` que descifraste en el Bloque 1, con media de 12.6%.

### 3.5 Los faltantes se propagaron solos

### 3.6 ¿Los elimino o los relleno?
Depende de **qué es** el dato que falta:
- **Ensayes que entran al balance** (`f`, `c`, `t`) → se eliminan. Rellenarlos  con la media es inventar masa que nunca se muestreó.
- **Variables de contexto** (`k80_Cab`) → se pueden imputar. No entran al balance.

Y el `subset` importa: una fila que perdió el concentrado de zinc **sigue sirviendo** para calcular plomo.

La tabla comparativa muestra lo siguiente sobre los diferentes métodos de imputación para `K80_Cab`:

- Original (con NaN): Teníamos una media de 54.99, una desviación estándar de 4.02 y 102 valores nulos.
- Imputación con la `Media` y la `Mediana`: Ambos métodos mantienen la media muy cercana al valor original (54.99). Esto es esperado, ya que se están reemplazando los NaNs con un valor central. Sin embargo, la desviación estándar disminuye a 3.94. Esto se debe a que al reemplazar los valores faltantes con un valor que está en el 'centro' de la distribución, se reduce la dispersión general de los datos. Ambos eliminan todos los NaNs.
- Imputación con `Ffill` (relleno hacia adelante): La media cambia ligeramente a 55.03 y la desviación estándar aumenta un poco a 4.04. Esto sugiere que los valores previos (hacia adelante) pueden ser un poco más dispersos o tener una tendencia que impacta la media y la desviación de manera diferente a un valor central.
- Imputación con `Bfill` (relleno hacia atrás): La media es 55.00 y la desviación estándar es 4.02, muy similar a los valores originales. Esto indica que los valores posteriores (hacia atrás) pueden tener una distribución más parecida a la de los datos existentes en promedio.


En resumen, la imputación con la media o la mediana tiende a subestimar la variabilidad (desviación estándar) al reemplazar los valores faltantes con un valor constante. Los métodos Ffill y Bfill pueden preservar mejor la variabilidad y la tendencia de los datos si hay una correlación temporal, pero también pueden introducir un sesgo si los valores faltantes representan cambios reales en los datos.

---
# Bloque 4 — ¿Quién lo está haciendo mejor?

### 4.0 Tu umbral de revisión
Mismo principio del Bloque 2: variable arriba, la cambias por la tuya.

### 4.1 ¿El turno de noche recupera menos?

### 4.2 ¿Y entre equipos?
La media sola no alcanza. Pide varias cosas de una vez.

**Mira las desviaciones, no solo las medias.** ¿De qué equipo te fiarías más?

### 4.3 Cruzar equipo y turno

### 4.4 ¿Cuál fue la recuperación del periodo?
Ya tienes la recuperación de cada turno. La pregunta ahora es una sola:**¿qué número le reportas a tu jefe?**

Antes de correr la celda, apuesta: ¿los tres van a dar lo mismo?

### 4.5 ¿Y si comparo equipos con cada método?

**La diferencia no es igual para todos los equipos.** ¿Qué implica eso para cualquier ranking que hayas hecho promediando?

---
# Bloque 5 — ¿Y cuál de todos es el correcto?

Ya tenemos dos números para la recuperación de zinc. Antes de reportar alguno,mira qué pasa **turno a turno** y no en el promedio.

### 5.1 El método por etapas no sobrevive al detalle

Recuperaciones **negativas** y **por encima de 100%**. Físicamente imposibles.El promedio se veía razonable. El detalle no lo es.La causa: cada etapa usa **un solo trazador**, y el error de ensaye se amplificaal encadenar dos cálculos.

### 5.2 Usar toda la información disponible

Un circuito con dos concentrados tiene tres incógnitas: las fracciones másicas
que van al concentrado de plomo, al de zinc y a colas finales.

Con **dos trazadores** (Pb y Zn) más el balance de masa se arma un sistema de 3×3:

$$
\begin{aligned}
c^{Pb}_{Pb}\,C_{Pb} + c^{Zn}_{Pb}\,C_{Zn} + t_{Pb}\,T &= f_{Pb} \\
c^{Pb}_{Zn}\,C_{Pb} + c^{Zn}_{Zn}\,C_{Zn} + t_{Zn}\,T &= f_{Zn} \\
C_{Pb} + C_{Zn} + T &= 1
\end{aligned}
$$

Mismo `np.linalg.solve` del Episodio 1.

### Cómo se leen las matrices A y b

El sistema es **A · x = b**, donde `x = [C_Pb, C_Zn, T]` son las tres fracciones
másicas que buscamos.

**La matriz A (3×3) — cada fila es una ecuación de balance:**

| | va al Conc Pb | va al Conc Zn | va a Colas | |
|---|---|---|---|---|
| **balance de Pb** | `Conc_Pb_Pb` | `Conc_Zn_Pb` | `Col_Zn_Pb` | → alimenta `Cab_Pb` |
| **balance de Zn** | `Conc_Pb_Zn` | `Conc_Zn_Zn` | `Col_Zn_Zn` | → alimenta `Cab_Zn` |
| **balance de masa** | `1` | `1` | `1` | → total `1` |

Cada columna es un **producto** (a dónde puede irse la masa) y cada fila es un
**elemento que se conserva**. Las dos primeras filas dicen "el plomo que entra por
cabeza tiene que aparecer repartido entre los tres productos"; lo mismo para el zinc.
La tercera fila dice que las tres fracciones suman 1: nada se pierde ni se inventa.

**El vector b — el lado derecho, lo que entró por cabeza:**

Ley de plomo en la cabeza, ley de zinc en la cabeza, y el 1 que cierra el balance
de masa.

**La diferencia entre dos productos y tres productos está aquí:** dos productos
resuelve una ecuación con **un** trazador. Tres productos usa **dos** trazadores (Pb
y Zn) más la restricción de que todo suma 1. Más ecuaciones sobre los mismos datos =
el error de cada ensaye pesa menos. Por eso la dispersión baja.

### 5.3 Primero un turno, para ver la mecánica

### 5.4 Ahora los todos los turnos de una sola llamada

Compara `C_Pb` con el `y1` que calculaste en el Bloque 3 (2.78%).Mismo split másico, dos métodos, dos números.

### 5.5 Los tres números, lado a lado

### 5.6 ¿Y qué tan de acuerdo están los dos métodos legítimos?

---
### El problema que queda abierto

Tres de cada cuatro turnos discrepan por más de 5 puntos entre dos métodos
que **ambos son correctos**.

Tres productos es **mejor**: un tercio de la dispersión y ningún valor imposible.
Pero no es **correcto**. Los ensayes traen error de muestreo y de laboratorio,
el balance no cierra, y ningún método puro lo arregla.

Ajustar los datos para que el balance cierre —cambiando lo mínimo posible y
respetando la confiabilidad de cada medición— se llama **reconciliación de balances**,
se resuelve con multiplicadores de Lagrange, y es el próximo episodio.